# Additional utilities

Small things around the methods: asking a language model, Wikipedia, posts from
Markdown, a tunnel for hooks.

In [ ]:
import os, tempfile
from pathlib import Path

# Examples stay off your real setup: secrets from environment variables
# (the `env` vault), an inbox and a dataset in a temporary directory.
scratch = Path(tempfile.mkdtemp())
os.environ["AMOORION_VAULT"] = "env"
os.environ["AMOORION_INBOX"] = str(scratch / "inbox.sqlite3")
os.environ["AMOORION_DATASET"] = str(scratch / "dataset")

In [ ]:
import json, threading
from http.server import BaseHTTPRequestHandler, HTTPServer


class FakeModel(BaseHTTPRequestHandler):
    """Speaks the chat completions API, as Ollama or a LiteLLM proxy would."""

    def do_POST(self):
        asked = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        said = asked["messages"][-1]["content"].splitlines()[0][:50]
        answer = {"choices": [{"message": {"content": f"(a model's words about: {said})"}}]}
        self.send_response(200)
        self.end_headers()
        self.wfile.write(json.dumps(answer).encode())

    def log_message(self, *args):
        pass


model_server = HTTPServer(("127.0.0.1", 0), FakeModel)
threading.Thread(target=model_server.serve_forever, daemon=True).start()
os.environ["AMOORION_SECRET_LLM_URL"] = f"http://127.0.0.1:{model_server.server_port}/v1"
os.environ["AMOORION_SECRET_LLM_MODEL"] = "fake"

### `compose`: one call for many models

Ollama, llama.cpp's server, a LiteLLM proxy, OpenAI, Claude and Hugging Face's router
all speak this API, so `llm/url`, `llm/model` and `llm/token` in the vault choose one.
Each secret stays in the vault; a local server needs no token.

In [ ]:
from amoorion import Amoorion

amo = Amoorion().use("dryrun")
amo.compose(prompt="Announce our new release", system="Write one short sentence.")

It chains with the rest in a workflow:

In [ ]:
steps = [
    {"method": "compose", "params": {"prompt": "Announce our new release"}},
    {"method": "publish", "params": {"text": "$last", "to": ["dryrun"]}},
]
amo.workflow(steps=steps)[1]["result"][0]["message"]["text"]

### Posts from Markdown

Front matter gives the title, tags, links and platforms:

In [ ]:
from amoorion.utilities import post_from_markdown

post, to = post_from_markdown("""---
title: Hello
tags: [news]
to: [dryrun]
---
Written once.
""")
print(post, to)
amo.publish(to=to, **post.model_dump())[0]["ok"]

### `wiki`

A Wikipedia summary, as a message to quote or to feed a workflow. It needs the
network, so it is shown here and not run:

```python
amo.wiki(title="Blacksmith", lang="en")["text"]
```

### A tunnel for hooks

Hooks need an address reachable from outside. With `pip install ngrok` and
`ngrok/token` in the vault:

```python
from amoorion.utilities import open_tunnel

listener = open_tunnel(8000, amo.vault)
print(listener.url())   # give this to GitHub, plus /hooks/github
```

Anyone with that address reaches the port, so only the API belongs behind it: its
methods need their token and its hooks verify themselves.